<a href="https://colab.research.google.com/github/arulbenjaminchandru/ai-engineer-june20/blob/main/Day_11_RAG_Theory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 11 — RAG: Teaching Claude to Answer From *Your* Documents

**The pipeline you will master today:** `chunk → embed → store → retrieve → generate`

---

## 🎬 Meet Meera (our scenario for Day 11 and Day 12)

Meera runs customer support at **NammaPay**, a small payments startup in Chennai.
Her team answers merchant questions all day using a **48-page Merchant Policy Handbook** (a PDF).

Monday morning, a merchant asks:

> *"A customer disputed a UPI payment. How many days do I have to respond?"*

A new agent guesses **"30 days"**. The handbook actually says **5 working days**.
The merchant misses the deadline and loses the dispute. Ouch.

Meera's idea: *"Can't Claude just read our handbook and answer these questions?"*

She tries pasting the whole 48-page PDF into every chat. It sort of works, but it is
slow, costly, and she has 200 more documents waiting. There has to be a better way.

**There is. It is called RAG — and by today evening you will have built it.**

---

## What you'll be able to do after today

- **Explain** RAG to anyone — even someone with zero AI background
- **Draw** the 5-stage pipeline (chunk → embed → store → retrieve → generate) from memory
- **Demonstrate** each stage with small, runnable code
- **Implement** a grounding prompt that stops Claude from making things up
- **Architect** the offline vs online halves of a RAG system like a pro

## Who this is for

AI engineers, architects, and enterprise developers — **no prior AI knowledge assumed**.
Every term is explained the moment it appears. If you can read Python, you can follow this.



## ⚙️ Setup (2 cells, 1 minute)

We install the Anthropic SDK and load your API key.

> **Before running:** in Colab, click the 🔑 **Secrets** icon in the left sidebar
> and add a secret named `MY_API_KEY` with your Anthropic API key as the value.

In [ ]:
# Install the official Anthropic SDK (quiet mode)
!pip install -q anthropic
print("Installed ✅")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 17.3 MB/s eta 0:00:00
Installed ✅


In [ ]:
# Load your key from Colab Secrets and create the Claude client
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("MY_API_KEY")

import anthropic
client = anthropic.Anthropic()          # reads the key from the environment
MODEL = "claude-haiku-4-5-20251001"     # Haiku: fast + cheap = perfect for learning

print("Claude client ready ✅")

Claude client ready ✅


---
# Section 1 — The Problem: Claude Has Never Read Your Handbook

🧠 **The idea:** Claude is brilliant, but it learned from public text on the internet
up to a certain date (its *training data*). Your company's private handbook was never
part of that. So when you ask about it, Claude has exactly two options:

1. **Admit it doesn't know** (good, but not helpful), or
2. **Guess** something that *sounds* right (dangerous — this is called a **hallucination**)

**Hallucination** = the model writes a confident, fluent answer that is simply not true.
It is not lying on purpose — it is completing a pattern, like autocomplete on your phone.

🔬 **Why it happens:** Claude has seen thousands of refund policies online. When you
ask about *NammaPay's* policy, it produces a *typical* policy — not *your* policy.
A typical-sounding wrong answer is worse than no answer, because people believe it.

💻 Let's see the problem live. NammaPay is fictional, so Claude cannot possibly know
its rules. Watch what happens:

In [ ]:
# Ask Claude about a company policy it has never seen
question = "How many days does a NammaPay merchant have to respond to a UPI payment dispute?"

reply = client.messages.create(
    model=MODEL,
    max_tokens=200,
    messages=[{"role": "user", "content": question}],
)

print(reply.content[0].text)

I don't have specific information about NammaPay's dispute resolution timeline in my current knowledge base.

To get accurate details about the merchant response period for UPI payment disputes on NammaPay, I'd recommend:

1. **Checking NammaPay's official website** or app - they should have a dedicated support/FAQ section
2. **Contacting NammaPay customer support** directly via phone, email, or in-app chat
3. **Reviewing the terms and conditions** you accepted when setting up your account

Generally, UPI payment disputes in India typically follow RBI guidelines, which usually allow merchants a specific timeframe (commonly 7-10 days) to respond, but the exact period can vary by payment provider.

Is there anything else about UPI disputes I can help clarify?


📊 **How to read this:** You will likely see one of two behaviours:

- Claude **admits it doesn't have NammaPay's rules** — honest, but Meera's merchant still has no answer, or
- Claude **describes a "typical" dispute window** (often quoting general RBI/NPCI norms) — which may not match NammaPay's actual rule at all.

Either way, the business problem is unsolved: **Claude cannot know what it was never shown.**

🧪 **Try this:** change the question to something about *your* company. Same problem.

---

## The fix, in one sentence

> **Don't teach Claude your documents. Show Claude the right page at the right time.**

💻 Watch what happens when we simply *include* the relevant policy text in the prompt:

In [ ]:
# The same question — but now we SHOW Claude the relevant handbook text
policy_text = """NammaPay Merchant Policy Handbook, Section 4.2 (Disputes):
When a customer disputes a UPI transaction, the merchant must respond with
evidence within 5 working days. If no response is received, the dispute is
automatically decided in the customer's favour."""

reply = client.messages.create(
    model=MODEL,
    max_tokens=200,
    messages=[{
        "role": "user",
        "content": f"Using this policy text:\n\n{policy_text}\n\nQuestion: {question}"
    }],
)

print(reply.content[0].text)

# Answer

According to NammaPay Merchant Policy Handbook, Section 4.2, a NammaPay merchant has **5 working days** to respond to a UPI payment dispute with evidence.


📊 **How to read this:** Now Claude answers **5 working days** — correct, and traceable
to Section 4.2. Nothing about Claude changed. **Only the context changed.**

🧠 That is the entire secret of RAG: *the model is frozen; the context is fresh.*

What we just did by hand — finding the right paragraph and pasting it in — is exactly
what RAG automates. The hard part is doing it for a 48-page PDF and *any* question.

⚠️ **Accuracy note:** People say RAG "gives Claude knowledge". Precisely: RAG changes
**what Claude reads at answer time** — it does not retrain or fine-tune the model, and
Claude remembers nothing afterwards. Every question starts fresh. Interviewers love
asking "Is RAG a form of fine-tuning?" The answer is a clear **no** — fine-tuning
changes the model's weights; RAG changes the model's *input*.

🤯 **Fun fact:** The name RAG comes from a 2020 research paper, *"Retrieval-Augmented
Generation for Knowledge-Intensive NLP Tasks"* by Patrick Lewis and colleagues at
Facebook AI Research (now Meta AI). Lewis has joked that they would have chosen a
nicer name if they'd known the acronym would take over the industry.

✅ **Quick check**

<details><summary>Why is pasting the whole 48-page handbook into every question a bad plan?</summary>

Three reasons: **cost** (you pay per token, every single question re-sends 48 pages),
**speed** (more input = slower answers), and **focus** (burying one relevant paragraph
in 47 pages of noise makes it easier for the model to miss it or blend in wrong
sections). RAG sends only the few paragraphs that matter.
</details>

---
# Section 2 — The RAG Pipeline: 5 Stages

🧠 **The librarian analogy** (remember this — we'll reuse it all day):

Imagine a library where nobody may take a book home (that's your document store).
A great librarian does five things:

1. **Chunk** — tears each book into labelled index cards (small pieces)
2. **Embed** — writes a "meaning code" on each card so similar cards can be found
3. **Store** — files all cards in a smart cabinet, organised by meaning
4. **Retrieve** — when you ask a question, pulls out the 3 most relevant cards
5. **Generate** — a smart assistant (Claude) reads *only those cards* and writes your answer

```
 OFFLINE (do once per document)          ONLINE (every question)
 ─────────────────────────────           ───────────────────────
 documents                                user question
    │                                        │
    ▼                                        ▼
 ① CHUNK  → small pieces                 embed the question
    │                                        │
    ▼                                        ▼
 ② EMBED  → each piece → vector          ④ RETRIEVE top-3 similar chunks
    │                                        │
    ▼                                        ▼
 ③ STORE  → vector database ────────────►⑤ GENERATE: Claude answers
                                            from those chunks only
```

🔬 **The one structural insight interviewers listen for:** the pipeline has two halves.
Stages 1–3 run **offline, once per document** (cheap to repeat, can run overnight).
Stages 4–5 run **online, on every question** (must be fast and cheap).
Keeping expensive work offline is what makes RAG affordable.

🏢 **Real world:** This is the architecture behind Notion AI Q&A (answers from your own
workspace pages), and enterprise assistants at banks and telecoms.
Anthropic's own docs describe this exact pattern for building Claude-powered
knowledge assistants.

**Remember this:** *"RAG = open-book exam. The model doesn't memorise the book —
it gets the right page open in front of it."*

❌ **Wrong:** "RAG stores knowledge inside the model." → ✅ **Right:** knowledge stays in
your database; the model only sees a few retrieved chunks per question.
❌ **Wrong:** "RAG needs a huge GPU to train." → ✅ **Right:** there is no training at all;
you can build one on a laptop.

🎤 **Interview Q:** *"Walk me through a RAG pipeline."* — Say it in one breath:
"Offline, we chunk documents, embed each chunk into a vector, and store text + vector
in a vector database. Online, we embed the user's question with the same model,
retrieve the top-k most similar chunks, and pass them to Claude with a grounding
prompt so the answer comes only from those chunks."

✅ **Quick check**

<details><summary>Which stages run on EVERY user question, and which run once per document?</summary>

Chunk, embed, store (stages 1–3) run **once per document** — offline.
Retrieve and generate (stages 4–5) run **on every question** — online.
(Small overlap: the *question* also gets embedded online, using the same embedding model.)
</details>

---
# Section 3 — Stage 1: Chunking (Cutting the Book Into Index Cards)

🧠 **What is it?** Chunking = splitting a big document into small pieces (chunks),
usually a paragraph or two each. Each chunk will later get its own "meaning code"
and be retrievable on its own.

**Why does it matter?** Two reasons:

- **Findability.** If the whole 48-page handbook is one big piece, a question about
  battery warranties matches it only vaguely — the one relevant paragraph is drowned
  out by 47 pages of other topics. Small chunks let the search zoom in.
- **Focus.** Claude answers best when it reads a few relevant paragraphs,
  not a wall of mostly-irrelevant text.

**When would I chunk less?** If your documents are already tiny (tweets, FAQ entries),
each one *is* a chunk. Chunking matters for anything longer than a page.

🔬 **The two dials you can turn:**

| Dial | Typical value | Effect |
|---|---|---|
| `chunk_size` | 200–600 tokens | smaller = laser-focused but less context; bigger = more context but blurrier search |
| `overlap` | 10–15% of chunk size | consecutive chunks share some text, so a sentence at a boundary isn't orphaned |

> A **token** is the unit models read — roughly 3/4 of an English word.
> Handy rule of thumb: **1 token ≈ 4 characters** of English text.

⚠️ **Accuracy note:** "1 token ≈ 4 characters" is a *rule of thumb* for plain English —
useful for sizing chunks, and that's how we'll use it. Precisely, tokenizers vary by
model and language: code, Tamil or Hindi text, and unusual words can take far more
tokens per character. Never bill a client from the ≈4 rule; measure with the real
tokenizer (the API's usage report tells you the true count).

In [ ]:
# Meera's handbook — a mini version we'll use all day (plain Python string)
HANDBOOK = """NammaPay Merchant Policy Handbook (mini edition).
Section 1, Settlements: NammaPay settles merchant payments on a T+1 basis. Money from
Monday's sales reaches your bank account on Tuesday. Settlement is free for UPI.
Section 2, Refunds: If a customer payment fails but money is deducted, it is
auto-refunded within 7 days. Merchants can issue manual refunds within 90 days.
Section 3, Disputes: When a customer disputes a UPI transaction, the merchant must
respond with evidence within 5 working days, or the dispute is decided in the
customer's favour. A fee of Rs 250 applies to each lost chargeback.
Section 4, KYC: Merchants must complete KYC with a PAN card and one address proof.
Accounts with incomplete KYC cannot receive settlements above Rs 50,000 per month.
Section 5, Support: Merchant support is available 9am to 9pm IST, all seven days,
via chat in the NammaPay dashboard."""

print(f"Handbook: {len(HANDBOOK)} characters ≈ {len(HANDBOOK)//4} tokens")

Handbook: 886 characters ≈ 221 tokens


## Strategy A — Fixed-size chunking (the simplest)

🧠 Cut every N characters, like slicing a loaf of bread — every slice identical,
no matter what's inside. Fast and predictable, but it can cut a sentence in half.

💻 Run it and *look at where it cuts*:

In [ ]:
def chunk_fixed(text, chunk_chars=300, overlap_chars=50):
    """Slide a fixed-size window through the text. Consecutive chunks overlap."""
    chunks, start = [], 0
    step = chunk_chars - overlap_chars      # how far the window moves each time
    while start < len(text):
        piece = text[start:start + chunk_chars].strip()
        if piece:
            chunks.append(piece)
        start += step
    return chunks

fixed_chunks = chunk_fixed(HANDBOOK, chunk_chars=300, overlap_chars=50)

print(f"{len(fixed_chunks)} chunks\n")
for i, c in enumerate(fixed_chunks[:3]):
    print(f"--- chunk {i} ({len(c)} chars) ---")
    print(c, "\n")

4 chunks

--- chunk 0 (300 chars) ---
NammaPay Merchant Policy Handbook (mini edition).
Section 1, Settlements: NammaPay settles merchant payments on a T+1 basis. Money from
Monday's sales reaches your bank account on Tuesday. Settlement is free for UPI.
Section 2, Refunds: If a customer payment fails but money is deducted, it is
auto-r 

--- chunk 1 (299 chars) ---
payment fails but money is deducted, it is
auto-refunded within 7 days. Merchants can issue manual refunds within 90 days.
Section 3, Disputes: When a customer disputes a UPI transaction, the merchant must
respond with evidence within 5 working days, or the dispute is decided in the
customer's favo 

--- chunk 2 (300 chars) ---
, or the dispute is decided in the
customer's favour. A fee of Rs 250 applies to each lost chargeback.
Section 4, KYC: Merchants must complete KYC with a PAN card and one address proof.
Accounts with incomplete KYC cannot receive settlements above Rs 50,000 per month.
Section 5, Support: Merchant su 

📊 **How to read this:** Look at the chunk endings — chunk 0 stops mid-sentence,
even mid-*word* ("...it is auto-r"), with the rest landing in the next chunk.
That's the fixed-size weakness: **the knife doesn't care about meaning.**
The 50-character overlap is the safety net: the cut sentence appears whole in
the *next* chunk, so it can still be found.

## Strategy B — Sentence-aware chunking (respect the full stops)

🧠 Instead of cutting every N characters, we add whole sentences to a chunk until
it's full, then start a new chunk. No sentence ever gets cut in half.

In [ ]:
import re

def chunk_sentences(text, max_chars=300):
    """Pack whole sentences into chunks of up to max_chars."""
    sentences = re.split(r"(?<=[.!?])\s+", text)   # split after . ! ?
    chunks, current = [], ""
    for s in sentences:
        if len(current) + len(s) > max_chars and current:
            chunks.append(current.strip())          # current chunk is full → save it
            current = ""
        current += s + " "
    if current.strip():
        chunks.append(current.strip())              # don't forget the last one
    return chunks

sent_chunks = chunk_sentences(HANDBOOK, max_chars=300)

print(f"{len(sent_chunks)} chunks\n")
for i, c in enumerate(sent_chunks[:3]):
    print(f"--- chunk {i} ({len(c)} chars) ---")
    print(c, "\n")

4 chunks

--- chunk 0 (216 chars) ---
NammaPay Merchant Policy Handbook (mini edition). Section 1, Settlements: NammaPay settles merchant payments on a T+1 basis. Money from
Monday's sales reaches your bank account on Tuesday. Settlement is free for UPI. 

--- chunk 1 (156 chars) ---
Section 2, Refunds: If a customer payment fails but money is deducted, it is
auto-refunded within 7 days. Merchants can issue manual refunds within 90 days. 

--- chunk 2 (228 chars) ---
Section 3, Disputes: When a customer disputes a UPI transaction, the merchant must
respond with evidence within 5 working days, or the dispute is decided in the
customer's favour. A fee of Rs 250 applies to each lost chargeback. 



📊 **How to read this:** Every chunk now ends at a full stop. Chunk sizes vary a
little — that's the trade: cleaner meaning, less uniform sizes.

## Strategy C — Recursive chunking (the industry favourite)

🧠 Try to split at the *biggest* natural boundary first — paragraphs. If a paragraph
is still too big, fall back to sentences. Still too big? Fall back to words.
It's called "recursive" because it keeps re-applying the same idea at smaller scales.
This is what LangChain's popular `RecursiveCharacterTextSplitter` does.

## Strategy D — Semantic chunking (the fancy one — know it, rarely need it)

🧠 Embed every sentence, then cut wherever the *meaning* suddenly changes
(the similarity between neighbouring sentences drops). Chunks align with real topic
boundaries — but it's slow and needs tuning. Mention it in interviews;
reach for it only when simpler strategies fail.

### Which one should you use?

| Strategy | Use when | Watch out |
|---|---|---|
| Fixed-size + overlap | default starting point, messy text | can cut sentences |
| Sentence-aware | clean prose | uneven sizes |
| Recursive | structured docs (headings, paragraphs) | slightly more code |
| Semantic | topic detection really matters | slow, needs tuning |

**Remember this:** *"Chunk size is the sharpest knob in RAG — tomorrow we'll prove it
by testing three sizes on a real PDF and watching the quality change."*

❌ **Wrong:** "Bigger chunks are always better — more context!" → ✅ **Right:** bigger
chunks blur the search; a question matches a big chunk weakly even when one paragraph
inside is perfect. It's a trade-off you *measure* (tomorrow's lab does exactly that).
❌ **Wrong:** "Overlap wastes storage, skip it." → ✅ **Right:** the ~10–15% storage cost
buys you protection against answers that straddle a boundary — nearly always worth it.

🎤 **Interview Q:** *"How do you choose chunk size?"* — "Start around 300–500 tokens
with 10–15% overlap, then **evaluate on real queries** — retrieval hit rate and answer
quality — and tune. It depends on the documents: dense policy text favours smaller
chunks, narrative text favours larger. The wrong answer is picking a number and never
measuring."

✅ **Quick check**

<details><summary>A merchant asks about the chargeback fee. The sentence "A fee of Rs 250 applies..." got cut in half between chunk 3 and chunk 4 (no overlap). What happens, and which dial fixes it?</summary>

Neither half-chunk matches the question strongly, so retrieval may miss it and Claude
never sees the fee → wrong answer or "I don't know". The **overlap** dial fixes it:
with overlap, the full sentence appears intact in at least one chunk.
</details>

---
# Section 4 — Stage 2: Embeddings (Writing the Meaning Code)

🧠 **What is it?** An **embedding** is a list of numbers (a *vector*) that captures the
*meaning* of a piece of text. Texts that mean similar things get similar numbers.

**The map analogy:** think of a giant map where every sentence gets a pin.
Sentences about refunds cluster in one neighbourhood, sentences about KYC in another.
An embedding is just the pin's coordinates. "How do I get my money back?" lands right
next to "Refund policy" — even though they share almost no words. **That's the magic:
embeddings match by meaning, not by matching words.**

🔬 Real embedding models output 384 to 3,000+ numbers per text. Humans can't picture
384 dimensions — but the *idea* works in 2. Let's build a tiny 2-D "meaning map" by
hand so you can see the machinery with your own eyes:

In [ ]:
import math

# A hand-made 2-D meaning map.  x = "about money/refunds",  y = "about identity/KYC"
# (Real models learn hundreds of dimensions automatically — same idea, more axes.)
mini_embeddings = {
    "Refunds take 7 days":             (0.9, 0.1),
    "How do I get my money back?":     (0.85, 0.15),   # different words, same meaning!
    "KYC needs a PAN card":            (0.1, 0.9),
    "What ID proof should I submit?":  (0.15, 0.85),
    "Support is open 9am to 9pm":      (0.4, 0.4),
}

def cosine_similarity(a, b):
    """1.0 = same direction (same meaning), 0 = unrelated."""
    dot   = sum(x*y for x, y in zip(a, b))
    len_a = math.sqrt(sum(x*x for x in a))
    len_b = math.sqrt(sum(x*x for x in b))
    return dot / (len_a * len_b)

query = "How do I get my money back?"
print(f"Query: {query!r}\n")
for text, vec in mini_embeddings.items():
    if text == query:
        continue
    sim = cosine_similarity(mini_embeddings[query], vec)
    print(f"  similarity {sim:.3f}  vs  {text!r}")

Query: 'How do I get my money back?'

  similarity 0.998  vs  'Refunds take 7 days'
  similarity 0.281  vs  'KYC needs a PAN card'
  similarity 0.342  vs  'What ID proof should I submit?'
  similarity 0.819  vs  'Support is open 9am to 9pm'


📊 **How to read this:** "How do I get my money back?" scores ~0.99 against
"Refunds take 7 days" — **zero words in common, nearly identical meaning-direction.**
Against the KYC sentence it scores much lower. Word-search could never do this;
meaning-search does it effortlessly.

🔬 **Cosine similarity** (the score we just computed) measures the *angle* between two
vectors: 1.0 = pointing the same way (same meaning), 0 = unrelated directions.
It ignores length, so a short text and a long text about the same topic still match.

### Where do real embeddings come from?

⚠️ **Accuracy note (a genuinely useful one):** **Anthropic does not ship its own
embedding model.** The Claude API has no `embeddings` endpoint — Anthropic's docs
officially point to **Voyage AI** for embeddings. So every Claude RAG system is a
two-vendor architecture by design: an embedding model (Voyage AI, or a free local
model like Sentence Transformers) does the *mechanical* meaning-matching, and Claude
does the *thinking* on top. Saying this unprompted in an interview signals you've
actually built one.

Common choices:

| Model | Dimensions | Cost | Notes |
|---|---|---|---|
| `all-MiniLM-L6-v2` (Sentence Transformers) | 384 | free, runs locally | our choice for tomorrow's lab — small and fast |
| `all-mpnet-base-v2` (Sentence Transformers) | 768 | free, runs locally | stronger, slower, bigger download |
| Voyage AI (`voyage-3.5` family) | ~1024 | paid API | Anthropic's recommended partner |
| OpenAI `text-embedding-3-small` | 1536 | paid API | common in mixed stacks |

🐛 **Gotcha you WILL hit one day:** the question and the documents must be embedded by
the **same model**. Different models draw *different maps* — coordinates from one are
meaningless on another. Symptom: retrieval returns garbage with no error message.

❌ **Wrong:** "Each number in the vector means something, like dimension 7 = 'anger'."
→ ✅ **Right:** meaning is spread across all dimensions together; individual numbers
are not human-interpretable.
❌ **Wrong:** "More dimensions = better quality." → ✅ **Right:** more dimensions = more
storage and compute; quality depends on the model's training, not the count.

**Remember this:** *"An embedding is a pin on a meaning map; similar meanings are
neighbours — that's what makes search-by-meaning possible."*

🤯 **Fun fact:** the famous demo that made word vectors a sensation (from Google's
2013 word2vec work): **king − man + woman ≈ queen**. Meaning arithmetic, with plain
vector addition. Modern sentence embeddings are the same idea, grown up.

✅ **Quick check**

<details><summary>Why does "How do I get my money back?" match "Refunds take 7 days" in embedding search but score terribly in a keyword search?</summary>

They share almost no words (keyword search sees nothing to match), but the embedding
model has learned from millions of sentences that both express the *refund* concept,
so their vectors point the same way → high cosine similarity.
</details>

---
# Section 5 — Stage 3: Store (The Smart Filing Cabinet)

🧠 **What is it?** A **vector database** stores three things per chunk:
the **text** (for Claude to read later), the **vector** (for meaning-search),
and **metadata** (labels like source file and chunk number, for filtering and citing).

**Why a special database?** A normal database finds *exact* matches ("give me row
where id=7"). A vector database finds *nearest neighbours* ("give me the 3 stored
vectors closest in meaning to this one") — and does it fast even with millions of chunks.

🏢 **Real world options:** **ChromaDB** (open-source, runs inside your notebook —
tomorrow's choice), **Pinecone** (managed cloud), **Weaviate**, **Qdrant**, and
**pgvector** (bolts vector search onto boring, reliable PostgreSQL — a favourite of
pragmatic enterprise teams).

💻 Let's store our hand-made 2-D vectors in a real vector database, right now.
ChromaDB installs in seconds and runs in memory:

In [ ]:
!pip install -q chromadb
print("ChromaDB installed ✅")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

In [ ]:
import chromadb

chroma = chromadb.Client()                      # in-memory database (RAM only)

collection = chroma.create_collection(
    name="mini_handbook",
    metadata={"hnsw:space": "cosine"},          # ← tell Chroma to use COSINE distance
)

texts   = list(mini_embeddings.keys())
vectors = [list(v) for v in mini_embeddings.values()]

collection.add(
    ids=[f"card_{i}" for i in range(len(texts))],       # unique ID per chunk
    documents=texts,                                    # the text Claude will read
    embeddings=vectors,                                 # the meaning coordinates
    metadatas=[{"source": "mini_handbook", "chunk_index": i} for i in range(len(texts))],
)

print(f"Stored {collection.count()} cards in the cabinet ✅")

Stored 5 cards in the cabinet ✅


⚠️ **Accuracy note (a mistake in most tutorials):** ChromaDB's **default** distance
is **L2 (straight-line distance), not cosine**. If you skip the
`metadata={"hnsw:space": "cosine"}` line, your scores are on a completely different
scale than the cosine similarities everyone talks about — with no error message.
We set cosine explicitly, always. (You can verify a collection's setting via its
configuration.) With cosine space, Chroma returns a *distance* where
**similarity = 1 − distance**.

🐛 **Gotcha:** Chroma collection names must be 3–512 characters (letters, digits,
`._-`, starting/ending alphanumeric). `create_collection(name="qa")` fails with a
validation error — a confusing first-run surprise.

✅ **Quick check**

<details><summary>Why store the raw text AND the vector, not just the vector?</summary>

The vector is only for *finding* the chunk — you can't reconstruct the text from it.
Claude needs the actual text to write the answer. Metadata rides along so you can
filter (e.g. only 2026 documents) and cite sources.
</details>

---
# Section 6 — Stage 4: Retrieve (Pulling the Right Cards)

🧠 **What is it?** Retrieval = embed the *question* with the same model, then ask the
vector database: "which stored chunks are closest in meaning?" It returns the **top-k**
(k = how many chunks you want — 3 is the classic default).

💻 Let's retrieve from our mini cabinet. New question, never stored:

In [ ]:
# A new question. We embed it with the SAME "map" (here: our hand-made 2-D scheme).
# "money back" is a refund-flavoured question → high x, low y on our map.
question_vector = [0.8, 0.2]

results = collection.query(
    query_embeddings=[question_vector],
    n_results=3,                                 # top-k: give me the 3 best cards
)

print("Question: 'A payment failed — when is the customer refunded?'\n")
for doc, dist in zip(results["documents"][0], results["distances"][0]):
    similarity = 1 - dist                        # cosine space: similarity = 1 - distance
    print(f"  similarity {similarity:.3f}  →  {doc!r}")

Question: 'A payment failed — when is the customer refunded?'

  similarity 0.998  →  'How do I get my money back?'
  similarity 0.991  →  'Refunds take 7 days'
  similarity 0.857  →  'Support is open 9am to 9pm'


📊 **How to read this:** The refund cards rank on top with the highest similarity;
the KYC card ranks last. The database compared *directions on the meaning map* —
no keywords involved.

### Choosing k (how many chunks to hand Claude)

| k | Effect |
|---|---|
| 1 | laser-focused; risky — one bad retrieval and Claude has nothing useful |
| **3** | **the safe default** — usually catches the answer plus helpful context |
| 5–8 | better for broad questions; more noise, more tokens, more cost |
| 20+ | Claude drowns in mostly-irrelevant text; slower and pricier |

⚠️ **Accuracy note:** retrieval returns the **most similar** chunks, not the **correct**
ones. If the answer isn't in your database, retrieval still cheerfully returns the
top-3 *least-bad* matches — with decent-looking scores. Similarity is relative, not a
guarantee of relevance. This is exactly why Stage 5 needs *grounding* (next section).

🎤 **Interview Q:** *"Your RAG system gives wrong answers. Where do you look first?"* —
"Retrieval, before generation. Print the retrieved chunks for failing questions: if
the right chunk isn't in the top-k, no prompt magic can fix the answer. Fix chunking,
k, or the embedding model first; only then tune the prompt."

🤯 **Fun fact:** vector databases can search millions of vectors in milliseconds using
an index called **HNSW** ("Hierarchical Navigable Small World" — the algorithm behind
Chroma, Qdrant and others). It's an *approximate* search: it trades a tiny bit of
accuracy for enormous speed — like asking a well-connected friend instead of
interviewing every person in the city.

✅ **Quick check**

<details><summary>Why must the question be embedded with the same model as the chunks?</summary>

Because each embedding model draws its own map. Coordinates only mean something on the
map that produced them. Mixing models compares pins from two different maps —
results look random, and nothing errors out to warn you.
</details>

---
# Section 7 — Stage 5: Generate, with Grounding (The Star of the Show)

🧠 **What is grounding?** Grounding = instructing Claude to answer **only from the
retrieved chunks** — and to say "I don't know" when they don't contain the answer.

The open-book exam rule: *you may only use what's on the open page. If the page
doesn't cover it, write "not covered" — don't improvise from memory.*

Without grounding, Claude helpfully blends retrieved text with its general training
knowledge — and you're back to confident wrong answers, now with false credibility
because *some* of the answer came from your real documents.

🔬 **The three rules of a good grounding prompt:**

1. **Restrict:** "Answer ONLY from the context below."
2. **Give an exit:** "If the context doesn't contain the answer, say: 'I don't have
   this information in the handbook.'" ← without an explicit exit, models feel
   pressured to produce *something*.
3. **Demand receipts:** "Cite the section you used." Citations make answers checkable
   — and forcing the model to point at its source visibly reduces drift.

💻 The real thing, live — grounded generation with Claude Haiku:

In [ ]:
GROUNDING_PROMPT = """You are NammaPay's merchant support assistant.

Rules:
1. Answer ONLY from the context below. Ignore your general knowledge.
2. If the context does not contain the answer, reply exactly:
   "I don't have this information in the handbook."
3. Quote or cite the section that supports your answer.

Context:
{context}"""

def ask_grounded(question, context):
    """One grounded call to Claude Haiku: context in the system prompt, question as the user turn."""
    reply = client.messages.create(
        model=MODEL,
        max_tokens=300,
        system=GROUNDING_PROMPT.format(context=context),
        messages=[{"role": "user", "content": question}],
    )
    print(f"Q: {question}")
    print(f"A: {reply.content[0].text}")
    print(f"   (input tokens: {reply.usage.input_tokens}, output: {reply.usage.output_tokens})\n")

# Pretend retrieval just returned these two chunks (tomorrow this happens automatically):
retrieved_context = """[Section 3, Disputes] When a customer disputes a UPI transaction, the
merchant must respond with evidence within 5 working days, or the dispute is decided
in the customer's favour. A fee of Rs 250 applies to each lost chargeback.

[Section 2, Refunds] If a customer payment fails but money is deducted, it is
auto-refunded within 7 days. Merchants can issue manual refunds within 90 days."""

# Test 1: the answer IS in the context
ask_grounded("How long do I have to respond to a dispute?", retrieved_context)

# Test 2: the answer is NOT in the context — will Claude admit it?
ask_grounded("What is NammaPay's office address in Chennai?", retrieved_context)

Q: How long do I have to respond to a dispute?
A: According to Section 3 (Disputes), you have **5 working days** to respond to a UPI transaction dispute with evidence. If you don't respond within this timeframe, the dispute is decided in the customer's favour.
   (input tokens: 200, output: 52)

Q: What is NammaPay's office address in Chennai?
A: I don't have this information in the handbook.

The context provided covers dispute procedures and refund policies, but does not contain information about NammaPay's office locations.
   (input tokens: 201, output: 39)



📊 **How to read this:**

- **Test 1** should answer *5 working days*, citing Section 3.
- **Test 2** is the important one: the address is nowhere in the context. A grounded
  system replies **"I don't have this information in the handbook."** If Claude
  invents an address, your grounding failed — tighten the rules.
- Notice the **token counts** printed: that's your actual cost per question, and the
  number that explodes if you retrieve too many chunks. Real numbers beat guesses.

**"I don't know" is a feature, not a failure.** In enterprise AI, a wrong answer costs
trust and sometimes money; an honest "not in the handbook" costs nothing and routes
the merchant to a human. Tomorrow we'll *measure* how often the system says it
correctly — a metric most teams forget to track.

⚠️ **Accuracy note:** grounding **reduces** hallucination dramatically; it does not
**eliminate** it. A prompt is an instruction, not a physical constraint — under
pressure (leading questions, near-miss context) models still occasionally blend in
outside knowledge. Production systems add layers: citation checking, a second
"is this answer supported by the context?" verification call, and human escalation.
Anyone who tells a client "RAG makes hallucination impossible" is setting them up.

❌ **Wrong:** "The system said 'I don't know' — it's broken." → ✅ **Right:** on an
out-of-scope question, "I don't know" is the *correct* output and you should test for it.
❌ **Wrong:** "Grounding guarantees truthful answers." → ✅ **Right:** grounding
guarantees *the instruction was given*; verification and evaluation make it reliable.

🏢 **Real world:** citation-first grounded answers are exactly how Perplexity built a
search product people trust, and why legal/medical AI tools show "sources" under every
answer. Enterprises often log every (question, retrieved chunks, answer) triple for
audit — in banking this can be a compliance requirement.

**Remember this:** *"Grounding turns 'Claude, what do you know?' into 'Claude, what
does the document say?' — a different question with a checkable answer."*

🎤 **Interview Q:** *"How do you stop a RAG system from hallucinating?"* —
"Three layers: a strict grounding prompt with an explicit 'I don't know' exit and
required citations; retrieval quality work so the right context is actually there;
and evaluation — an out-of-scope test set that measures how often the system
correctly declines. You can't claim grounding works if you never measured it."

✅ **Quick check**

<details><summary>Why does the grounding prompt explicitly provide the sentence to say when the answer is missing?</summary>

Two reasons: without a sanctioned exit, the model feels compelled to produce *some*
answer (that pressure produces hallucinations); and a fixed refusal phrase is easy to
detect in code, which lets you *measure* grounding accuracy automatically — tomorrow's
lab does exactly that.
</details>

---
# Section 8 — The Whole Pipeline in One Cell (Mini RAG, Live)

🧠 You've now seen all five stages separately. Watch them click together:
**retrieve from ChromaDB → build context → grounded Claude call.**
This is a complete, working RAG query — the same skeleton you'll wrap around a
real PDF tomorrow.

In [ ]:
def mini_rag(question, question_vector, k=2):
    """A complete RAG query in ~15 lines: retrieve → assemble context → generate."""

    # STAGE 4 — RETRIEVE: nearest chunks from the vector database
    results = collection.query(query_embeddings=[question_vector], n_results=k)
    chunks  = results["documents"][0]

    # Assemble the retrieved chunks into one context block
    context = "\n".join(f"[Card {i}] {c}" for i, c in enumerate(chunks))

    # STAGE 5 — GENERATE: grounded call to Claude Haiku
    reply = client.messages.create(
        model=MODEL,
        max_tokens=300,
        system=GROUNDING_PROMPT.format(context=context),
        messages=[{"role": "user", "content": question}],
    )

    print(f"Q: {question}")
    print(f"Retrieved: {chunks}")
    print(f"A: {reply.content[0].text}\n")

# On our 2-D map: refund question → high x;  KYC question → high y
mini_rag("A payment failed — when does the customer get the money back?", [0.85, 0.15])
mini_rag("Which documents do I need for KYC?",                            [0.1, 0.9])

Q: A payment failed — when does the customer get the money back?
Retrieved: ['How do I get my money back?', 'Refunds take 7 days']
A: Based on the handbook:

**Refunds take 7 days** (from Card 1).

When a payment fails, the customer should expect their money to be returned within 7 days.

Q: Which documents do I need for KYC?
Retrieved: ['KYC needs a PAN card', 'What ID proof should I submit?']
A: Based on the handbook, for KYC you need a **PAN card**.

This is mentioned in Card 0: "KYC needs a PAN card"

However, if you're looking for a complete list of all required ID proofs, Card 1 addresses "What ID proof should I submit?" but the specific details from that section aren't fully provided in my context. For a comprehensive answer about all acceptable documents, you may want to refer to Card 1 directly or contact support.



📊 **How to read this:** each question retrieved *different* cards (refund cards for
the first, KYC cards for the second), and each answer cites only what was retrieved.
The pipeline routed each question to the right knowledge automatically.
Tomorrow, a real embedding model replaces our hand-made coordinates — everything
else stays identical.

🧪 **Try this (a productive failure):** call
`mini_rag("What are NammaPay's support hours?", [0.1, 0.9])` — a support question with
deliberately *wrong* coordinates (KYC direction). Watch retrieval fetch KYC cards and
grounded Claude reply "I don't have this information." **Bad retrieval → honest
refusal, not a made-up answer.** That's the grounding safety net doing its job —
and it shows why retrieval quality, not the prompt, is usually what you fix first.

---
# Section 9 — The Architect's View

Every RAG system, from your notebook to a bank's, has the same skeleton:

```mermaid
flowchart TB
    subgraph OFFLINE["OFFLINE - once per document (can run overnight)"]
        A[Documents: PDFs, wikis, tickets] --> B[1. Chunk]
        B --> C[2. Embed each chunk]
        C --> D[(3. Vector DB: text + vector + metadata)]
    end
    subgraph ONLINE["ONLINE - every question (must be fast)"]
        Q[User question] --> E[Embed question - SAME model]
        E --> F[4. Retrieve top-k chunks]
        D --> F
        F --> G[5. Claude Haiku + grounding prompt]
        G --> H[Grounded answer + citations]
    end
```

**Who does what:**

| Component | Job | Typical choice | Failure smell |
|---|---|---|---|
| Chunker | cut documents well | your ~15 lines of Python | answers "half missing" |
| Embedding model | draw the meaning map | MiniLM / Voyage AI | retrieval looks random |
| Vector DB | store + fast nearest-neighbour search | Chroma / pgvector / Pinecone | slow or stale results |
| Claude Haiku | read chunks, write grounded answer | Messages API | ignores context, waffles |
| Grounding prompt | keep Claude inside the context | your system prompt | confident wrong answers |

**Failure points that produce NO error message** (the dangerous kind):

- Mixed embedding models between indexing and querying → silent garbage retrieval
- Default L2 vs intended cosine distance → scores on the wrong scale, silently
- Documents updated but never re-indexed → confident answers from *stale* text
- Scanned/image PDFs → text extraction returns empty strings, pipeline "works" on nothing

**What's still missing before production** (Day 12 builds the core; a real deployment
adds): re-indexing when documents change; access control (not every employee may see
every chunk); logging of question → chunks → answer for audit; an evaluation set that
runs on every change; rate limiting and cost monitoring; and a human escalation path
for "I don't know" cases.

### The claude.ai bridge (no-code equivalent)

| You built (API) | Claude product equivalent |
|---|---|
| chunk + embed + store + retrieve | **Projects** on claude.ai: upload files to project knowledge, Claude answers from them |
| grounding prompt | Project instructions ("answer only from the uploaded documents") |
| your Python Q&A loop | the chat itself |

If a client needs 20 documents searchable by 5 people, a claude.ai Project may be the
right answer — no code at all. The API pipeline earns its keep at scale: thousands of
documents, custom logic, audit trails, or embedding into your own product. Knowing
*when not to build* is an architect skill too.

---
# Section 10 — Beyond the Basics (Know These Names)

Four upgrades you'll meet in real systems and interviews. Today: recognise them.

**1. Hybrid retrieval.** Meaning-search sometimes misses exact codes ("error E-402",
"model A12"). Hybrid runs *both* embedding search and classic keyword search (BM25 —
the ranking algorithm inside Elasticsearch), then merges the ranked lists (a standard
merge trick is called Reciprocal Rank Fusion). Default in serious production systems.

**2. Re-ranking.** Fast retrieval grabs the top-50; a slower, smarter model re-scores
those 50 and keeps the best 5. A speed/precision two-stage combo — the same pattern
web search engines use.

**3. Query expansion.** The user asks "return policy", the handbook says "refund
policy". Generate a few rewrites of the query (Claude can do this), search with all
of them, merge the results. Cheap recall boost.

**4. Multi-hop retrieval.** "How does our refund policy affect Q3 revenue?" needs the
policy *and* the finance report. Retrieve → let Claude ask a follow-up → retrieve
again → synthesise. This is where RAG starts becoming an *agent*.

🎤 **Interview Q:** *"Dense retrieval keeps missing product codes — what do you do?"*
— "Classic dense-retrieval weakness: exact identifiers. Add hybrid retrieval — BM25
alongside embeddings, merge with rank fusion — and re-rank if precision still lags."


---
# 🛠️ Mini Project Spec — "NammaPay Handbook Assistant" (you build it on Day 12)

**Business case:** Meera's 12 support agents answer ~400 merchant questions/day from
the handbook. Target: instant, cited, grounded answers — and honest refusals routed
to humans.

**Architecture:** exactly today's diagram — pdfplumber → chunker → MiniLM embeddings
→ ChromaDB (cosine) → top-3 retrieval → Claude Haiku with grounding prompt.

**Build steps (all Day 12):**
1. Upload any PDF and extract clean text
2. Chunk with overlap; embed; store in ChromaDB
3. Grounded Q&A function with citations
4. **The experiment:** index the same PDF at 3 chunk sizes and *measure* which wins
5. Grounding test: out-of-scope questions must be refused

**Definition of done (checklist):**
- [ ] Any text PDF can be uploaded and queried
- [ ] Answers cite chunk/section numbers
- [ ] Scorecard printed comparing 3 chunk sizes on the same questions
- [ ] ≥ 4 of 5 out-of-scope questions correctly get "I don't have this information"
      ← *note: a check that a FAILURE behaves correctly is part of done*
- [ ] You can explain the whole flow to a teammate in 2 minutes

Today's theory made claims (chunk size matters; grounding works). **Tomorrow we don't
trust claims — we print numbers.**

---
# 📝 Session Summary

Claude cannot answer from documents it has never seen, and when pushed it may produce
confident, wrong answers (hallucination). RAG fixes this not by retraining the model
but by changing what it reads at answer time: offline, documents are **chunked** into
small pieces, each piece is **embedded** into a vector that captures its meaning, and
text + vector + metadata are **stored** in a vector database; online, the user's
question is embedded with the *same* model, the top-k most similar chunks are
**retrieved**, and Claude Haiku **generates** an answer under a grounding prompt that
restricts it to the retrieved context, provides an explicit "I don't have this
information" exit, and demands citations. Chunk size and overlap are the sharpest
quality knobs (we measure them tomorrow); retrieval returns *similar* — not
guaranteed-*correct* — chunks, which is why grounding plus evaluation, not either
alone, is what makes RAG trustworthy.

# ✅ What You Learned Today

You can now:
- [ ] Explain hallucination and why it happens (pattern completion, not lying)
- [ ] Draw the 5-stage pipeline and split it into offline vs online halves
- [ ] Implement fixed-size and sentence-aware chunking in plain Python
- [ ] Explain embeddings with the meaning-map analogy and compute cosine similarity
- [ ] Store and query a real vector database (ChromaDB, cosine space)
- [ ] Write a grounding prompt with the three rules: restrict, exit, cite
- [ ] Run a complete grounded RAG query against Claude Haiku
- [ ] Say what RAG does NOT guarantee — and how production systems close the gap

# 🗂️ AI Architect Cheat Sheet — RAG

**Definitions in one line each**
- **RAG** — retrieve relevant document chunks, let the model answer only from them
- **Chunk** — a small piece of a document (a few hundred tokens)
- **Embedding** — a vector (list of numbers) encoding a text's meaning
- **Vector DB** — stores text+vector+metadata; finds nearest-by-meaning fast
- **Top-k** — how many chunks retrieval returns (default 3)
- **Grounding** — prompt rules restricting answers to retrieved context
- **Hallucination** — fluent, confident, false output

**Numbers worth memorising**

| Number | Meaning |
|---|---|
| 200–600 tokens | typical chunk size range |
| 10–15% | typical overlap |
| 3 | default top-k |
| 384 / 768 | MiniLM / MPNet embedding dimensions |
| 1 token ≈ 4 chars | English rule of thumb (estimate only) |
| 2020 | RAG paper, Lewis et al., Facebook AI |

**Claude API quick reference**
```python
import anthropic
client = anthropic.Anthropic()                      # key from env: ANTHROPIC_API_KEY
reply = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=300,
    system=grounding_prompt_with_context,           # rules + retrieved chunks
    messages=[{"role": "user", "content": question}],
)
print(reply.content[0].text, reply.usage.input_tokens)
```

**ChromaDB quick reference**
```python
import chromadb
col = chromadb.Client().create_collection("docs", metadata={"hnsw:space": "cosine"})
col.add(ids=ids, documents=texts, embeddings=vectors, metadatas=metas)
res = col.query(query_embeddings=[qvec], n_results=3)   # similarity = 1 - distance
```

**Decision table**

| Situation | Reach for |
|---|---|
| few docs, few users, no code | claude.ai Project |
| many docs / custom logic / audit | API pipeline (this course) |
| misses exact codes & IDs | hybrid retrieval (add BM25) |
| retrieval noisy | re-ranking, or smaller k |
| user words ≠ document words | query expansion |

# ⏱️ 5-Minute Revision Guide

1. LLMs can't answer from documents they never saw — and may hallucinate instead.
2. Hallucination = confident pattern-completion, not intentional lying.
3. RAG = show the model the right text at answer time; no retraining involved.
4. Pipeline: chunk → embed → store → retrieve → generate.
5. Chunk/embed/store run offline, once per document; retrieve/generate run per question.
6. Chunking splits documents so search can zoom in on the right paragraph.
7. Chunk size 200–600 tokens, overlap 10–15% — starting points, then measure.
8. Overlap protects sentences that straddle chunk boundaries.
9. Strategies: fixed-size, sentence-aware, recursive (industry default), semantic.
10. Embedding = meaning-pin on a map; similar meaning → similar vector.
11. Cosine similarity measures the angle between vectors: 1 = same meaning.
12. Same embedding model for documents AND questions — always.
13. Anthropic has no embedding API — pair Claude with Voyage AI or Sentence Transformers.
14. Vector DB stores text + vector + metadata; ChromaDB default is L2 — set cosine explicitly.
15. Retrieval returns most-*similar* chunks, never guaranteed-*correct* ones.
16. Top-k = 3 is the safe default; more k = more context but more noise and cost.
17. Grounding prompt: restrict to context + explicit "I don't know" exit + citations.
18. "I don't know" on out-of-scope questions is correct behaviour — test for it.
19. Grounding reduces hallucination; evaluation and verification make it reliable.
20. Debug retrieval before prompts: if the right chunk isn't retrieved, nothing downstream can save you.

# 🎤 Interview Preparation Notes

**Q1. What is RAG and why use it over fine-tuning?**
"RAG retrieves relevant document chunks and lets the model answer only from them.
Versus fine-tuning: no training cost, updates are instant (re-index, not retrain),
answers are traceable to sources, and access control stays in the database.
Fine-tuning shapes *behaviour and style*; RAG supplies *facts*. They're complements,
not rivals."

**Q2. Walk me through the pipeline.**
"Offline: chunk documents, embed each chunk, store text+vector+metadata in a vector
DB. Online: embed the question with the same model, retrieve top-k similar chunks,
generate with a grounding prompt. Offline is per-document; online is per-question."

**Q3. How do you pick chunk size?**
"Start 300–500 tokens, 10–15% overlap; then evaluate on real queries and tune.
Small chunks retrieve precisely but can lose surrounding context; big chunks carry
context but blur retrieval. The key point: it's an empirical knob — measure it."

**Q4. How do you prevent hallucination in RAG?**
"Layers: grounding prompt with an explicit refusal exit and citations; retrieval
quality so the right context is present; an out-of-scope test set to measure correct
refusals; optionally a verification pass checking the answer against the context.
And honesty: reduction, not elimination."

**Q5 (architecture). The system answers correctly but slowly and expensively. Ideas?**
"Check k and chunk size first — context tokens dominate cost; the usage report shows
input tokens per call. Use Haiku rather than a bigger model — grounded reading is
exactly what small models do well. Cache frequent questions. Keep embeddings local
(MiniLM) if latency allows. Measure before and after."

**Q6 (FDE-style). A client says 'we can't send our documents to a cloud AI.' Options?**
"Chunks are only sent at question time — and only the retrieved few, not the corpus;
that alone reassures many clients. Beyond that: redact sensitive fields before
indexing, filter by access level via metadata, or discuss deployment options that
keep data in the client's environment. Also worth naming: Anthropic's API terms —
by default — don't train models on business customers' data; verify current terms
with the client's legal team."


# 📚 Assignment

**Beginner** — On paper, draw the 5-stage pipeline from memory and label the
offline/online halves. Then explain RAG to a friend using the librarian analogy —
out loud, no notes.

**Intermediate** — Extend `chunk_fixed` to report, for each chunk, whether it starts
or ends mid-sentence. What fraction of chunks are "clean" at 300 chars vs 600 chars?

**Advanced** — Implement recursive chunking: split on paragraphs first (`\n\n`);
any paragraph longer than `max_chars` falls back to sentence packing. Compare its
output with both strategies from today on the mini handbook.

**Project prep** — Find 2–3 real PDFs you care about (a manual, a syllabus, a policy)
and bring them tomorrow. Write down 5 questions per PDF *and* the answers you'd
expect — that's tomorrow's evaluation set.

# 🧪 Assessment

**Part A — Multiple choice (10)**

1. RAG improves answers by:
   a) fine-tuning the model on your documents b) changing what the model reads at answer time c) increasing model size d) lowering temperature
2. Correct pipeline order:
   a) embed→chunk→store→generate→retrieve b) chunk→embed→store→retrieve→generate c) store→chunk→embed→retrieve→generate d) chunk→store→embed→generate→retrieve
3. Which stages run once per document (offline)?
   a) retrieve+generate b) chunk+embed+store c) embed+retrieve d) all five
4. An embedding is:
   a) a compressed copy of the text b) a keyword list c) a vector encoding meaning d) an encrypted string
5. Cosine similarity of ~0.95 between two texts means:
   a) same author b) same length c) very similar meaning d) identical words
6. Overlap between chunks exists to:
   a) reduce storage b) protect sentences at chunk boundaries c) speed up embedding d) confuse retrieval
7. ChromaDB's default distance metric is:
   a) cosine b) dot product c) L2 (Euclidean) d) Hamming
8. Retrieval guarantees the returned chunks are:
   a) factually correct b) the most similar available c) grammatically clean d) recent
9. A good grounding prompt includes:
   a) restrict to context, explicit "I don't know" exit, citations b) a request to be creative c) higher max_tokens d) multiple languages
10. Anthropic's recommended embedding partner is:
    a) OpenAI b) Cohere c) Voyage AI d) Anthropic's own embedding endpoint

**Part B — Short answer (5)**

11. Why must the same embedding model be used at index time and query time?
12. Give two reasons pasting an entire 48-page PDF into every prompt is worse than RAG.
13. Explain why "I don't know" can be the *correct* output, and how you'd measure it.
14. What breaks silently if your PDF is a scanned image? How would you notice?
15. Name the two halves of the RAG architecture and one component in each.

**Part C — Scenario (3)**

16. Users ask about "late fees"; the handbook says "grace period charges"; retrieval
    misses it. Diagnose the failure and give two fixes in the order you'd try them.
17. Your RAG bot answered a question about employee vacation policy — but only refund
    documents are indexed. What went wrong, in which stage, and what's the first fix?
18. After re-uploading updated handbooks, answers still quote old prices. Walk through
    the pipeline and say exactly where the staleness lives and the fix.

# 🔑 Answer Key

**MCQ:** 1-b · 2-b · 3-b · 4-c · 5-c · 6-b · 7-c · 8-b · 9-a · 10-c

**Short answers (gist):**
11. Each model defines its own vector space ("map"); vectors are only comparable within
one space. Mixing models silently produces near-random retrieval.
12. Cost (every question re-pays for the whole PDF in input tokens), speed, and focus
(the relevant paragraph is buried in noise). Any two.
13. When the context lacks the answer, refusing is correct — it prevents hallucination
and routes to humans. Measure with out-of-scope questions: % correctly refused.
14. Text extraction returns empty/near-empty strings; no error is raised; the system
indexes nothing and refuses everything (or retrieves junk). Notice by logging extracted
character counts per page and alerting on near-zero.
15. Offline/ingestion (chunker, embedder, vector DB writes) and online/query
(query embedding, retriever, Claude + grounding prompt).

**Scenarios (fix order matters):**
16. Vocabulary mismatch (user words ≠ document words) in the retrieval stage.
Fix order: (1) query expansion — cheap, no re-index; (2) hybrid retrieval adding BM25
— catches exact/near terms; then consider a domain-tuned embedding model.
17. The generation stage answered from training knowledge → grounding failure (weak or
missing grounding prompt). First fix: enforce the strict grounding prompt with an
explicit refusal exit — then re-test with out-of-scope questions.
18. The vector DB still holds chunks of the *old* documents — staleness lives in the
store stage. Fix: re-run ingestion (re-chunk, re-embed, upsert/replace), and add
re-indexing to the document-update workflow so it can't be forgotten.

# 🔗 Sources

- Lewis et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks* (2020) — arxiv.org/abs/2005.11401
- Anthropic docs — Messages API & model IDs: docs.claude.com
- Anthropic docs — Embeddings (Voyage AI guidance): docs.claude.com/en/docs/build-with-claude/embeddings
- ChromaDB docs — collections & distance metrics: docs.trychroma.com
- Sentence-Transformers model cards (MiniLM, MPNet): sbert.net
- Mikolov et al., word2vec (2013) — arxiv.org/abs/1301.3781

---
**🚀 Tomorrow (Day 12):** everything above, welded onto a *real PDF you choose* —
plus the experiment this theory promised: three chunk sizes, one scorecard, no guessing.